# Canonical Model 00: Conceptual Model and Build

This notebook begins the master canonical-example set. Every notebook uses the same irregular DISV/Voronoi model: four hydrostratigraphic layers, two lakes, diverted SFR, UZF, mountain-front RCH, two pumping wells, and two aquifer-specific DRN seepage slopes.

> **Purpose:** establish one trusted model whose physics, observations, calibration, particle tracking, parallelization, and visualization can all be tested together.

### Conceptual cross section

| Layer | Hydrogeologic role | Expected response |
|---|---|---|
| 1 | Upper unconfined aquifer | Rapid UZF, pond, stream, and seepage response |
| 2 | Lower unconfined aquifer | Pumping response and regional flow |
| 3 | Aquitard | Muted vertical communication |
| 4 | Confined aquifer | Confined pumping cone and slope seepage |

In [1]:
from pathlib import Path
import simple_modflow as mf
from canonical_notebook_style import notebook_header

notebook_header('00', 'Conceptual Model and Build', 'One model, one contract, every workflow.')

workspace = Path('../artifacts/canonical_master')
config = mf.CanonicalModelConfig.validation()  # use CanonicalModelConfig() for >=10,000 cells
model = mf.build_canonical_model(workspace / 'gwf', config=config, name='canonical_master')
# The contract checks grid type, hydrostratigraphy, packages, targets, and refinement.
mf.CANONICAL_MODEL_CONTRACT.validate(model)
model.regions.region_summary()

VoronoiGrid initializing.
Voronoi grid initialized.
Imported 1 features from ..\artifacts\canonical_master\gwf\inputs\north_lake.gpkg
Imported 1 features from ..\artifacts\canonical_master\gwf\inputs\south_lake.gpkg
Generating connections (rectangular mode) for lake 0
getting connectivity properties (iac, ja, cl12, hwva, nja)
Generating connections (rectangular mode) for lake 1


C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:302: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(
C:\Users\lukem\Python\mf-env\.venv\Lib\site-packages\flopy\mf6\mfmodel.py:278: DeprecationWarning: This method is for internal use only and will be deprecated.
  warnings.warn(


,name,kind,category,package,layer,num_cells,tags
4,all_lakes,region,boundary,lak,0,57,[lak]
6,all_streams,region,boundary,sfr,0,44,[sfr]
1,confined_seepage_slope,region,seepage,drn,3,12,[]
8,infiltration_pond,region,infiltration,uzf,0,4,"[mounding, visualization]"
2,lake_zone_lake_0,region,boundary,lak,0,28,[lak]
3,lake_zone_lake_1,region,boundary,lak,0,29,[lak]
5,sfr_group_stream,region,boundary,sfr,0,44,[sfr]
0,unconfined_seepage_slope,region,seepage,drn,0,9,[]
7,uzf_active,region,boundary,uzf,0,300,[uzf]


In [2]:
# A canonical build is only useful when its complete package topology runs.
success, report = model.run_simulation()
assert success, '\n'.join(report[-30:])
mf.canonical_head_signals(model)

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_gwf...
  writing model canonical_master...
    writing model name file...
    writing package disv...
    writing package ic...
    writing package npf...
    writing package sto...
    writing package chd...
INFORMATION: maxbound in ('', 'chd', 'dimensions') changed to 80 based on size of stress_period_data
    writing package ghb...
INFORMATION: maxbound in ('', 'ghb', 'dimensions') changed to 80 based on size of stress_period_data
    writing package rch...
    writing package wel...
INFORMATION: maxbound in ('', 'wel', 'dimensions') changed to 2 based on size of stress_period_data
    writing package drn...
INFORMATION: maxbound in ('', 'drn', 'dimensions') changed to 21 based on size of stress_period_data
    writing package lak...
    writing package sfr...
    writing package mvr...
    writing package uzf...
    writing package gwf_obs...
    writing packa

{'frame_count': 12,
 'spatial_range_by_layer': array([25.28007775, 24.83916724, 26.55003242, 25.77632158]),
 'temporal_range_by_layer': array([4.18465   , 4.35183253, 4.18465   , 5.28168224]),
 'maximum_drawdown_by_layer': array([2.092325  , 2.25950753, 2.092325  , 3.18935724])}

## Reading the result

Look for a strong regional gradient across every layer, transient movement in the unconfined aquifer, and a distinct confined-aquifer response. Continue to **01 · Packages and Observations** to inspect how each physical feature becomes a named, queryable model object.

In [3]:
import simple_modflow
from simple_modflow.modflow.mf6.simulation.base import SimulationBase
model: SimulationBase

In [4]:
model.packages.lak.results.q.map().plot()

In [11]:
model.packages.sfr.results.stage.get()


,model,package,kstpkper,per,reach,layer,cell,rlen,distance_start,distance_mid,distance_end,stage
0,canonical_master,sfr,"(0, 0)",0,0,0,320,47.661836,0.000000,23.830918,47.661836,128.067270
1,canonical_master,sfr,"(0, 0)",0,1,0,321,35.219876,47.661836,65.271774,82.881712,127.842863
2,canonical_master,sfr,"(0, 0)",0,2,0,301,53.045715,82.881712,109.404569,135.927427,127.841903
3,canonical_master,sfr,"(0, 0)",0,3,0,281,34.029385,135.927427,152.942120,169.956812,127.841702
4,canonical_master,sfr,"(0, 0)",0,4,0,261,11.929892,169.956812,175.921758,181.886704,127.840424
...,...,...,...,...,...,...,...,...,...,...,...,...
523,canonical_master,sfr,"(1, 5)",5,39,0,237,32.074092,2415.752699,2431.789745,2447.826790,110.335635
524,canonical_master,sfr,"(1, 5)",5,40,0,217,59.012368,2447.826790,2477.332974,2506.839158,110.335312
525,canonical_master,sfr,"(1, 5)",5,41,0,197,54.972080,2506.839158,2534.325198,2561.811238,109.719402
526,canonical_master,sfr,"(1, 5)",5,42,0,198,129.257583,2561.811238,2626.440030,2691.068821,108.619287
